# 第9章：推理优化

## 本章目标
- 理解 KV Cache 的原理和实现
- 掌握模型量化的基本方法（4-bit, 8-bit）
- 对比 FP16 vs 量化模型的推理速度和显存占用

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers bitsandbytes accelerate
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## 推理优化速览

模型训练完成后，推理效率是生产部署的关键。

三大优化方向：
1. **KV Cache**：避免重复计算 attention 的 key/value，生成速度提升数倍
2. **量化 (Quantization)**：降低模型精度（FP16 → INT8/INT4），减少显存和加速推理
3. **其他**：Speculative Decoding、模型蒸馏等（本教程不深入）

参考：[llama.cpp](https://github.com/ggerganov/llama.cpp), [vLLM](https://github.com/vllm-project/vllm)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    input_ids = tokenizer("Hello, how are you", return_tensors="pt").input_ids.cuda()

    # 无 KV Cache
    def generate_no_cache(model, input_ids, max_new_tokens=50):
        generated = input_ids.clone()
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = model(generated)
            logits = outputs.logits
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
        return generated

    # 有 KV Cache
    def generate_with_cache(model, input_ids, max_new_tokens=50):
        generated = input_ids.clone()
        past_key_values = None
        for _ in range(max_new_tokens):
            with torch.no_grad():
                if past_key_values is None:
                    outputs = model(generated, use_cache=True)
                else:
                    outputs = model(generated[:, -1:], past_key_values=past_key_values, use_cache=True)
            logits = outputs.logits
            past_key_values = outputs.past_key_values
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
        return generated

    # 对比速度
    torch.cuda.synchronize()
    t0 = time.time()
    out_no_cache = generate_no_cache(model, input_ids, max_new_tokens=50)
    torch.cuda.synchronize()
    t_no_cache = time.time() - t0

    torch.cuda.synchronize()
    t0 = time.time()
    out_cache = generate_with_cache(model, input_ids, max_new_tokens=50)
    torch.cuda.synchronize()
    t_cache = time.time() - t0

    print(f"No cache: {t_no_cache:.3f}s")
    print(f"KV cache: {t_cache:.3f}s")
    print(f"Speedup: {t_no_cache/t_cache:.1f}x")
else:
    print("需要 GPU（Colab T4）运行 KV cache 对比实验。")

In [ ]:
if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig

    # FP16 基准
    torch.cuda.empty_cache()
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    fp16_mem = torch.cuda.memory_allocated() / 1e9
    print(f"FP16 显存: {fp16_mem:.3f} GB")

    # 4-bit 量化
    del model_fp16
    torch.cuda.empty_cache()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto"
    )
    q4_mem = torch.cuda.memory_allocated() / 1e9
    print(f"4-bit 显存: {q4_mem:.3f} GB")
    print(f"压缩比: {fp16_mem/q4_mem:.1f}x")
else:
    print("需要 GPU（Colab T4）运行量化对比实验。")

In [ ]:
if torch.cuda.is_available():
    def benchmark_inference(model, tokenizer, prompt, num_runs=10, max_new_tokens=50):
        inputs = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        times = []
        for _ in range(num_runs):
            torch.cuda.synchronize()
            t0 = time.time()
            with torch.no_grad():
                model.generate(inputs, max_new_tokens=max_new_tokens)
            torch.cuda.synchronize()
            times.append(time.time() - t0)
        return sum(times) / len(times)

    prompt = "Explain quantum computing in simple terms."

    # 重新加载 FP16 模型（之前被删除了）
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    t_fp16 = benchmark_inference(model_fp16, tokenizer, prompt)
    t_4bit = benchmark_inference(model_4bit, tokenizer, prompt)

    print(f"FP16 avg: {t_fp16:.3f}s")
    print(f"4-bit avg: {t_4bit:.3f}s")
    print(f"速度比: {t_fp16/t_4bit:.2f}x")
else:
    print("需要 GPU（Colab T4）运行 benchmark。")

## 练习

1. 尝试 8-bit 量化（`load_in_8bit=True`），对比 4-bit 和 8-bit 的效果和速度
2. 增大 `max_new_tokens` 到 200，观察 KV cache 的加速效果是否更明显
3. 在更大的模型（如 Qwen2.5-1.5B）上重复实验，观察量化带来的显存节省

## 延伸阅读

- [llama.cpp](https://github.com/ggerganov/llama.cpp) — GGUF 格式量化推理引擎
- [bitsandbytes](https://github.com/TimDettmers/bitsandbytes) — GPU 量化库
- [GPTQ 论文](https://arxiv.org/abs/2210.17323) — Post-training 量化方法
- [vLLM](https://github.com/vllm-project/vllm) — 高性能推理引擎（PagedAttention）